In [28]:
import statsmodels
import torch
import torch.nn as nn
import torch.optim as optim
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
import pandas as pd

In [6]:
data = pd.read_csv("../data/processed_vol_features.csv", index_col=0, parse_dates=True)

In [7]:
data.head()

,Close,log_return,realized_vol5,realized_vol10,realized_vol21,vol_of_vol5,parkinson,gk_vol,rv_lag1,rv_lag5,rv_lag21,vix,vix_change,mom_5d,mom_21d,target_rv
Date,,,,,,,,,,,,,,,,
2020-02-11,306.383240,0.001732,0.100497,0.149352,0.133205,0.810295,0.078104,0.010284,0.125554,0.197924,0.072707,15.18,0.009309,0.018842,0.029320,0.080312
2020-02-12,308.357208,0.006422,0.080312,0.149226,0.132990,0.774494,0.076915,0.009709,0.100497,0.208508,0.080210,13.74,-0.094862,0.013700,0.028876,0.084153
2020-02-13,308.028229,-0.001067,0.084153,0.150795,0.132874,0.345296,0.077899,0.015213,0.080312,0.208490,0.075032,14.15,0.029840,0.009222,0.029348,0.056829
2020-02-14,308.521759,0.001601,0.056829,0.095641,0.132839,0.404818,0.077526,0.010428,0.084153,0.125535,0.071916,13.68,-0.033216,0.016256,0.028672,0.054499
2020-02-18,307.726685,-0.002580,0.054499,0.101056,0.131008,0.308663,0.078824,0.012618,0.056829,0.125554,0.078410,14.83,0.084064,0.006126,0.017557,0.060254


In [8]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1018 entries, 2020-02-11 to 2025-12-26
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Close           1018 non-null   float64
 1   log_return      1018 non-null   float64
 2   realized_vol5   1018 non-null   float64
 3   realized_vol10  1018 non-null   float64
 4   realized_vol21  1018 non-null   float64
 5   vol_of_vol5     1018 non-null   float64
 6   parkinson       1018 non-null   float64
 7   gk_vol          1018 non-null   float64
 8   rv_lag1         1018 non-null   float64
 9   rv_lag5         1018 non-null   float64
 10  rv_lag21        1018 non-null   float64
 11  vix             1018 non-null   float64
 12  vix_change      1018 non-null   float64
 13  mom_5d          1018 non-null   float64
 14  mom_21d         1018 non-null   float64
 15  target_rv       1018 non-null   float64
dtypes: float64(16)
memory usage: 135.2 KB


In [9]:
series = data["realized_vol21"].dropna()

split  = int(len(series) * 0.85)   # 85% train, 15% test

train  = series.iloc[:split]
test   = series.iloc[split:]

print(f"Train: {len(train)} obs  "
      f"({train.index[0].date()} → {train.index[-1].date()})")
print(f"Test : {len(test)}  obs  "
      f"({test.index[0].date()} → {test.index[-1].date()})")

Train: 865 obs  (2020-02-11 → 2024-11-26)
Test : 153  obs  (2024-11-27 → 2025-12-26)


In [11]:
import numpy as np

In [12]:
p, d, q = 2, 0, 1    

history = list(train)
predictions = []

print(f"Running walk-forward forecast "
      f"ARIMA({p},{d},{q}) on {len(test)} steps...")

for i, actual in enumerate(test):
    # Fit on all available history
    model = ARIMA(history, order=(p, d, q))
    fit   = model.fit()

    # Forecast 1 step ahead
    pred  = fit.forecast(steps=1)[0]
    predictions.append(pred)

    # Add actual value to history (walk-forward)
    history.append(actual)

    if i % 50 == 0:
        print(f"  Step {i+1}/{len(test)}  "
              f"pred={pred:.2f}  actual={actual:.2f}")

predictions = np.array(predictions)
actuals  = test.values

Running walk-forward forecast ARIMA(2,0,1) on 153 steps...


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


  Step 1/153  pred=0.14  actual=0.14


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

  Step 51/153  pred=0.52  actual=0.52


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

  Step 101/153  pred=0.10  actual=0.10


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

  Step 151/153  pred=0.11  actual=0.10


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [17]:
#evaluation
mae  = mean_absolute_error(actuals, predictions)
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mape = np.mean(np.abs((actuals - predictions) /
               (actuals + 1e-8))) * 100

# Directional accuracy — did vol go up/down correctly?
actual_dir = np.sign(np.diff(actuals))
pred_dir   = np.sign(np.diff(predictions))
dir_acc    = np.mean(actual_dir == pred_dir) * 100

print(f"  ARIMA({p},{d},{q}) — Baseline Results")
print(f"  MAE             : {mae:.4f}")
print(f"  RMSE            : {rmse:.4f}")
print(f"  MAPE            : {mape:.2f}%")
print(f"  Directional Acc : {dir_acc:.2f}%")

  ARIMA(2,0,1) — Baseline Results
  MAE             : 0.0113
  RMSE            : 0.0259
  MAPE            : 7.07%
  Directional Acc : 53.95%


In [ ]:
# #NaiveBaseline
# class NaiveBaseline:
#     """Predict last known value (or last N values repeated)"""
#     def predict(self, X):
#         # X shape: (batch, seq_len, features)
#         return X[:, -1, :]  # repeat last timestep

# # Evaluate
# with torch.no_grad():
#     naive = NaiveBaseline()
#     preds = naive.predict(test_X)  # (N, 1)
#     mae = torch.mean(torch.abs(preds - test_y.squeeze(-1)))
#     print(f"Naïve MAE: {mae:.4f}")


# ## Moving Average Baseline
# class MovingAverageBaseline:
#     def __init__(self, window=7):
#         self.window = window

#     def predict(self, X):
#         # X: (batch, seq_len, 1)
#         return X[:, -self.window:, :].mean(dim=1)  # avg of last `window` steps

# ma = MovingAverageBaseline(window=7)
# preds = ma.predict(test_X)

### Deep Learning Forecasting

In [24]:
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from sklearn.preprocessing import MinMaxScaler

In [25]:
class VolDataset(Dataset):
    def __init__(self, dataframe, feature_cols, target_col, seq_len, horizon = 1, scaler_X = None, scaler_y = None, fit_scalers=False):
        self.df = dataframe
        self.seq_len = seq_len
        self.horizon = horizon
        X_raw = self.df[feature_cols].values.astype(np.float32)
        y_raw = self.df[[target_col]].values.astype(np.float32)
        if fit_scalers:
            self.scaler_X = MinMaxScaler()
            self.scaler_y = MinMaxScaler()
            X_scaled = self.scaler_X.fit_transform(X_raw)
            y_scaled = self.scaler_y.fit_transform(y_raw)
        else:
            self.scaler_X = scaler_X
            self.scaler_y = scaler_y
            X_scaled = self.scaler_X.transform(X_raw)
            y_scaled = self.scaler_y.transform(y_raw)
        self.X, self.y = self._make_sequences(X_scaled, y_scaled)
        
    def _make_sequences(self, X, y):
        X_seq, y_seq = [], []
        total = len(X) - self.seq_len - self.horizon + 1
        for i in range(total):
            X_seq.append(X[i:i+self.seq_len])
            y_seq.append(y[i+self.seq_len+self.horizon-1])
        return (torch.tensor(np.array(X_seq), dtype=torch.float32), torch.tensor(np.array(y_seq), dtype=torch.float32))
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # row = self.df.iloc[idx]
        # data = row.drop("realized_vol21").values.astype(np.float32)
        # target = row["realized_vol21"].astype(np.float32)
        return self.X[idx], self.y[idx]

In [26]:
feature_cols = [col for col in data.columns if col != "realized_vol21"]
target_col = "realized_vol21"
n = len(data)
train_end = int(n * 0.7)
val_end = int(n * 0.85)
train_data = data.iloc[:train_end]
val_data = data.iloc[train_end:val_end]
test_data = data.iloc[val_end:]

In [29]:
train_dataset = VolDataset(train_data, feature_cols, target_col, seq_len=30, horizon=1, fit_scalers=True)
val_dataset = VolDataset(val_data, feature_cols, target_col, seq_len=30, horizon=1, 
                         scaler_X=train_dataset.scaler_X, scaler_y=train_dataset.scaler_y)
test_dataset = VolDataset(test_data, feature_cols, target_col, seq_len=30, horizon=1, 
                          scaler_X=train_dataset.scaler_X, scaler_y=train_dataset.scaler_y)

In [30]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

### MLP Forecaster

### LSTM Forecaster

In [23]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # Use last time step
        return out.squeeze()

In [ ]:
model = LSTMModel(input_size=data.shape[1]-1, hidden_size=64, num_layers=2)

### CNN-LSTM Forecaster
using news around option 